# CMES-86051 — Fleiss' κ и human_agreement.json (Раздел 9, конец)

Считает **фактический** Fleiss' κ по `human_ratings_raw_anonymized.csv`
(200 выходов × 3 раторa, 3 критерия, шкала 1–5). Число `κ=0.76` из рукописи
здесь ни на что не влияет — оно не задаётся заранее.

Pooling method, используемый ниже (нужно явно указать в статье): каждый из
трёх критериев обрабатывается как независимый "item" для целей pooled κ —
т.е. pooled κ считается на 200×3=600 псевдо-items, каждый оценён теми же
3 raters по шкале 1-5. Per-criterion κ считается отдельно на 200 items
каждый. Если вы хотите другой способ pooling (например, majority-category
per output усреднённый по критериям), нужно менять только функцию
`build_pooled_matrix`.


In [ ]:
import csv
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

CATEGORIES = [1, 2, 3, 4, 5]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def load_ratings(path):
    with open(path, encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))

CONFIG = {
    "RATINGS_CSV": "human_eval/human_ratings_raw_anonymized.csv",
    "OUTPUT_DIR": "human_eval",
}


In [ ]:
# ============================ FLEISS' KAPPA (стандартная формула) ============================

def fleiss_kappa(category_counts: list[list[int]]) -> float:
    n_items = len(category_counts)
    n_raters = sum(category_counts[0])
    assert all(sum(row) == n_raters for row in category_counts), "не все items оценены одинаковым числом raters"

    n_categories = len(category_counts[0])
    p_j = [sum(row[j] for row in category_counts) / (n_items * n_raters) for j in range(n_categories)]

    P_i = []
    for row in category_counts:
        P_i.append((sum(c * c for c in row) - n_raters) / (n_raters * (n_raters - 1)))
    P_bar = sum(P_i) / n_items
    P_e = sum(p * p for p in p_j)

    if P_e == 1.0:
        return 1.0  # вырожденный случай: полное согласие по определению p_e
    return (P_bar - P_e) / (1 - P_e)


In [ ]:
# ============================ ПОСТРОЕНИЕ МАТРИЦ ============================

rows = load_ratings(CONFIG["RATINGS_CSV"])
print(f"Прочитано строк: {len(rows)} (ожидается 600 = 200 outputs x 3 raters)")
assert len(rows) == 600, "human_ratings_raw_anonymized.csv должен содержать ровно 600 строк"

def build_matrix_for_criterion(rows: list[dict], criterion_key: str) -> tuple[list[list[int]], list[str]]:
    by_output: dict[str, list[int]] = {}
    for row in rows:
        by_output.setdefault(row["output_blind_id"], []).append(int(row[criterion_key]))
    output_ids = sorted(by_output)
    matrix = []
    for output_id in output_ids:
        ratings = by_output[output_id]
        assert len(ratings) == 3, f"{output_id}: ожидалось 3 оценки, найдено {len(ratings)}"
        counts = [ratings.count(cat) for cat in CATEGORIES]
        matrix.append(counts)
    return matrix, output_ids

def build_pooled_matrix(rows: list[dict]) -> list[list[int]]:
    # Каждый (output_blind_id, criterion) как отдельный псевдо-item -> 600 items x 3 raters
    pooled = []
    for criterion_key in ("criterion_1", "criterion_2", "criterion_3"):
        matrix, _ = build_matrix_for_criterion(rows, criterion_key)
        pooled.extend(matrix)
    return pooled

per_criterion_kappa = {}
for criterion_key, label in [
    ("criterion_1", "criterion_1_faithfulness"),
    ("criterion_2", "criterion_2_coverage"),
    ("criterion_3", "criterion_3_usefulness"),
]:
    matrix, output_ids = build_matrix_for_criterion(rows, criterion_key)
    kappa = fleiss_kappa(matrix)
    per_criterion_kappa[label] = kappa
    print(f"{label}: n_items={len(matrix)}, kappa={kappa:.4f}")

pooled_matrix = build_pooled_matrix(rows)
pooled_kappa = fleiss_kappa(pooled_matrix)
print(f"pooled (600 pseudo-items = 200 outputs x 3 criteria): kappa={pooled_kappa:.4f}")


In [ ]:
# ============================ СОХРАНИТЬ human_agreement.json ============================

agreement = {
    "created_utc": utc_now(),
    "n_outputs": 200,
    "n_raters": 3,
    "categories": CATEGORIES,
    "matrix_shape_per_criterion": "200 items x 5 categories, counts of 3 raters per item",
    "pooling_method": (
        "pooled kappa computed by stacking the three criteria as independent "
        "pseudo-items: 200 outputs x 3 criteria = 600 pseudo-items, each rated "
        "by the same 3 raters on a 1-5 scale. This is one explicit pooling "
        "choice among several possible (e.g. majority-vote-per-output averaged "
        "across criteria) and must be stated exactly this way in the manuscript."
    ),
    "per_criterion_kappa": per_criterion_kappa,
    "pooled_kappa": pooled_kappa,
    "note": (
        "This value is the actually observed Fleiss' kappa from "
        "human_ratings_raw_anonymized.csv. It was NOT set or adjusted to match "
        "any target value from the manuscript (e.g. 0.76)."
    ),
}

out_path = Path(CONFIG["OUTPUT_DIR"]) / "human_agreement.json"
out_path.write_text(json.dumps(agreement, ensure_ascii=False, indent=2), encoding="utf-8")
print(out_path)
agreement
